
# 02 - Bronze Auto Loader Ingestion
## 05 - Bronze Layer Validation

Validates table existence, row counts, schema correctness, and key field nulls for all bronze tables.

In [0]:
%run ../01_setup/00_config

In [0]:
import logging
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

In [0]:
for table in [bronze_market_prices_table, bronze_weather_table, bronze_grid_events_table]:
    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {table}").collect()[0]["row_count"]
    if row_count > 0:
        print(f"Check passed: {table} has {row_count} rows.")
    else:
        raise ValueError(f"Check failed: {table} is empty.")

## Expected schemas and key columns

In [0]:
expected_schemas = {
    bronze_market_prices_table: {
        "required_columns": {
            "event_date", "region", "market_type",
            "price_eur_mwh", "volume_mwh", "source_system", "last_updated_ts"
        },
        "key_columns": ["event_date", "region", "price_eur_mwh", "volume_mwh"]
    },
    bronze_weather_table: {
        "required_columns": {
            "event_date", "region", "temperature_c",
            "wind_speed_kmh", "precipitation_mm", "source_system", "last_updated_ts"
        },
        "key_columns": ["event_date", "region"]
    },
    bronze_grid_events_table: {
        "required_columns": {
            "event_id", "event_date", "region", "asset_id",
            "event_type", "severity", "source_system", "last_updated_ts"
        },
        "key_columns": ["event_id", "event_date", "region"]
    }
}

## Pre-ingestion - source file checks

In [0]:
print(catalog_name)
print(raw_schema)
print(landing_volume)

In [0]:
source_paths = {
    bronze_market_prices_table: f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/market_prices",
    bronze_weather_table:       f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/weather",
    bronze_grid_events_table:   f"/Volumes/{catalog_name}/{raw_schema}/{landing_volume}/grid_events",
}

for table, path in source_paths.items():
    try:
        files = dbutils.fs.ls(path)
        csv_files = [f for f in files if f.name.endswith(".csv")]
        if len(csv_files) == 0:
            raise ValueError(f"[SOURCE]  No CSV files found in: {path}")
        log.info(f"[SOURCE]  {table} - {len(csv_files)} source file(s) found in {path}")
    except Exception as e:
        raise ValueError(f"[SOURCE]  Cannot access source path {path}: {e}")

## Post-ingestion - bronze table checks

In [0]:
validation_passed = True
results = []

for table, config in expected_schemas.items():
    log.info(f"--- Validating: {table} ---")

    # table existence
    try:
        df = spark.table(table)
        log.info(f"[EXISTS]  {table} found.")
    except AnalysisException:
        log.error(f"[MISSING] Table not found: {table}")
        validation_passed = False
        results.append({"table": table, "check": "existence", "status": "FAILED"})
        continue

    # row count
    row_count = df.count()
    if row_count > 0:
        log.info(f"[ROWS]    {table} has {row_count} rows.")
        results.append({"table": table, "check": "row_count", "status": "PASSED", "value": row_count})
    else:
        log.error(f"[ROWS]    {table} is empty.")
        validation_passed = False
        results.append({"table": table, "check": "row_count", "status": "FAILED", "value": 0})

    # schema correctness
    actual_columns  = set(df.columns)
    required_columns = config["required_columns"]
    missing_columns  = required_columns - actual_columns

    if missing_columns:
        log.error(f"[SCHEMA]  {table} missing columns: {missing_columns}")
        validation_passed = False
        results.append({"table": table, "check": "schema", "status": "FAILED", "missing": str(missing_columns)})
    else:
        log.info(f"[SCHEMA]  {table} schema OK.")
        results.append({"table": table, "check": "schema", "status": "PASSED"})

    # key field nulls
    for key_col in config["key_columns"]:
        null_count = df.filter(F.col(key_col).isNull() | (F.trim(F.col(key_col)) == "")).count()
        if null_count > 0:
            log.error(f"[NULLS]   {table}.{key_col} has {null_count} null/empty values.")
            validation_passed = False
            results.append({"table": table, "check": f"nulls_{key_col}", "status": "FAILED", "value": null_count})
        else:
            log.info(f"[NULLS]   {table}.{key_col} OK.")
            results.append({"table": table, "check": f"nulls_{key_col}", "status": "PASSED"})

## Validation summary

In [0]:
summary_df = spark.createDataFrame(results)
display(summary_df)

if not validation_passed:
    raise ValueError("Bronze validation failed. Check the summary above for details.")
else:
    log.info("All bronze validation checks passed.")